# 07 CPU GPU Benchmarks

Select **Runtime → Change runtime type → NVIDIA GPU**, then run all cells
in a fresh session. Each notebook installs its own immutable public package
checkout and verifies every input. No Python-source upload is required.

Package source: `Favioleiva/gpugwr@7298c5629a48e7b875739f117a27d2e00f2512de`. These links become usable only
after owner-authorized publication. Local tests are not Colab verification.
A failed GPU request or numerical comparison stops execution visibly.
Save the executed notebook and download the output ZIP as evidence.


In [ ]:
import json
import os
import subprocess
import sys
import tempfile
from pathlib import Path

SOURCE_COMMIT = "7298c5629a48e7b875739f117a27d2e00f2512de"
REPOSITORY = "https://github.com/Favioleiva/gpugwr.git"
RUN = Path(tempfile.mkdtemp(prefix="gpugwr-"))
CHECKOUT = RUN / "source"
OUTPUT = RUN / "outputs"
OUTPUT.mkdir()
subprocess.run(["git", "init", str(CHECKOUT)], check=True)
subprocess.run(
    [
        "git",
        "-C",
        str(CHECKOUT),
        "remote",
        "add",
        "origin",
        REPOSITORY,
    ],
    check=True,
)
subprocess.run(
    [
        "git",
        "-C",
        str(CHECKOUT),
        "fetch",
        "--depth",
        "1",
        "origin",
        SOURCE_COMMIT,
    ],
    check=True,
)
subprocess.run(
    [
        "git",
        "-C",
        str(CHECKOUT),
        "checkout",
        "--detach",
        "FETCH_HEAD",
    ],
    check=True,
)
os.environ["CUPY_CACHE_DIR"] = str(RUN / "cupy-cache")
with (OUTPUT / "bootstrap.log").open("w", encoding="utf-8") as log:
    completed = subprocess.run(
        [
            sys.executable,
            str(CHECKOUT / "Colab/bootstrap.py"),
            "--checkout",
            str(CHECKOUT),
            "--commit",
            SOURCE_COMMIT,
            "--output",
            str(OUTPUT / "bootstrap.json"),
        ],
        stdout=log,
        stderr=subprocess.STDOUT,
        text=True,
    )
print((OUTPUT / "bootstrap.log").read_text("utf-8")[-12000:])
completed.check_returncode()


In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
from threadpoolctl import threadpool_limits
from gpuGWR import GWRSpatialCache, gpuGWR
from gpugwr.notebook_measurement import environment, measure
from gpugwr.reproducibility import (
    ReproductionAssets,
    export_geography,
    model_options,
    validate_arrays,
    validate_fit,
    write_json,
    zip_outputs,
)

assets = ReproductionAssets(CHECKOUT, RUN / "data-cache")
runtime = environment()
assert runtime["gpu"]["available"], (
    "A genuine GPU is required; no CPU fallback."
)
runtime["source_commit"] = SOURCE_COMMIT
write_json(OUTPUT / "environment.json", runtime)
print(json.dumps(runtime, indent=2))


## CPU/GPU benchmark protocol

Test all five empirical datasets; Brazil uses historical model1 here
(notebook 05 reproduces all four). Each engine gets one fresh-process fit
and three warmed repetitions. Fresh GPU workers receive empty on-disk
CuPy caches. Record process wall time separately from fit wall time.
BLAS is limited to one thread; CPU/GPU ordering alternates each repeat.

Model construction, fitting, transfers, synchronization and pool cleanup
are timed; data acquisition, validation and exporting are excluded from
fit time. Cold subprocess total time additionally includes imports,
loading, validation and exports. The environment probe warms the parent
CUDA context, but never the fresh worker. Every timed result is validated.
Report medians of measured trials, without extrapolating to other GPUs.
Whole-device sampled memory is a lower bound, not an exact peak; private
CuPy pool high-water measurements are retained in GPU configuration.


In [ ]:
from time import perf_counter

DATASETS = ["georgia", "columbus", "baltimore", "boston", "brazil"]
REPEATS = 3
records = []
for dataset in DATASETS:
    frame, specification = assets.load_case(dataset)
    model = (
        specification["models"][0] if dataset == "brazil" else None
    )
    options = model_options(specification, model=model)
    reference = assets.acquire(
        (model or specification)["reference"]
    )
    for engine in ("cpu", "gpu"):
        cold_output = OUTPUT / dataset / (engine + "_cold")
        worker_environment = os.environ.copy()
        worker_environment["CUPY_CACHE_DIR"] = str(
            RUN / (dataset + "_" + engine + "_cold_cache")
        )
        started = perf_counter()
        completed = subprocess.run(
            [
                sys.executable,
                "-m",
                "gpugwr.notebook_benchmark",
                "--checkout",
                str(CHECKOUT),
                "--cache",
                str(assets.cache),
                "--dataset",
                dataset,
                "--engine",
                engine,
                "--output",
                str(cold_output),
            ],
            env=worker_environment,
            text=True,
            capture_output=True,
        )
        cold_output.parent.mkdir(parents=True, exist_ok=True)
        (cold_output.parent / (engine + "_cold.log")).write_text(
            completed.stdout + completed.stderr, encoding="utf-8"
        )
        completed.check_returncode()
        timing = json.loads(
            (cold_output / "timing.json").read_text("utf-8")
        )
        timing["subprocess_total_seconds"] = (
            perf_counter() - started
        )
        records.append(timing)

    with threadpool_limits(limits=1):
        for engine in ("cpu", "gpu"):
            warmup = gpuGWR(
                frame,
                **options,
                engine=engine,
                store_matrices=model is None,
            ).fit()
            validate_fit(
                warmup,
                reference,
                OUTPUT
                / dataset
                / (engine + "_warmup_validation.json"),
            )
        for repetition in range(REPEATS):
            order = (
                ("cpu", "gpu")
                if repetition % 2 == 0
                else ("gpu", "cpu")
            )
            for engine in order:
                result, timing = measure(
                    lambda: gpuGWR(
                        frame,
                        **options,
                        engine=engine,
                        store_matrices=model is None,
                    ).fit(),
                    engine,
                )
                label = f"{engine}_warm_{repetition + 1}"
                validate_fit(
                    result,
                    reference,
                    OUTPUT / dataset / (label + "_validation.json"),
                )
                timing.update(
                    {
                        "dataset": dataset,
                        "phase": "warm",
                        "repeat": repetition + 1,
                        "configuration": result.configuration,
                        "model_options": options,
                    }
                )
                records.append(timing)
                if repetition == 0:
                    result.export(OUTPUT / dataset / label)
                    export_geography(
                        result,
                        frame,
                        specification,
                        assets,
                        OUTPUT / dataset / label,
                    )
                    print(result.summary())
                    display(result.local_results().head())
    write_json(OUTPUT / "benchmark_trials.json", records)
table = pd.DataFrame(
    [
        {
            "dataset": item["dataset"],
            "engine": item["engine"],
            "phase": item["phase"],
            "wall_seconds": item["wall_seconds"],
            "effective_batch_size": item["configuration"]
            .get("gpu_execution", {})
            .get("batch_size"),
            "sampled_peak_mib": item[
                "sampled_peak_device_memory_mib"
            ],
        }
        for item in records
    ]
)
table.to_csv(OUTPUT / "benchmark_trials.csv", index=False)
summary = (
    table[table.phase == "warm"]
    .groupby(["dataset", "engine"])
    .wall_seconds.agg(["count", "median", "min", "max"])
)
summary.to_csv(OUTPUT / "benchmark_summary.csv")
display(summary)


In [ ]:
archive = zip_outputs(OUTPUT)
print(f"Evidence archive: {archive}")
try:
    from google.colab import files
except ImportError:
    from IPython.display import FileLink

    display(FileLink(str(archive)))
else:
    files.download(str(archive))
